# Contraste: ¿qué puede detectar mi método y dónde sí vive una ganancia?

**Autor:** Samuel  
**Institución:** Tecnológico de Monterrey  
**Área:** Pairs trading en Forex  
**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud, Eventos, Datos-Diarios, Momentum-FX

Los notebooks anteriores no encontraron una ganancia neta operable. Aquí respondemos dos preguntas que dan valor a esos negativos:
1. **¿Qué tamaño de ganancia podría detectar mi método?** (un resultado "no significativo" solo es informativo si el método tenía poder para detectar algo).
2. **¿Encuentra el mismo procedimiento una ganancia donde la literatura dice que existe?** (control positivo), y en qué se diferencia de lo que probamos.

## Estructura de este Notebook
1. Fase 0: Librerías y parámetros
2. Fase 1: Poder estadístico (qué ganancia mínima puede detectar cada muestra)
3. Fase 2: Control positivo con ETFs (momentum): datos y regla
4. Fase 3: ETFs, resultados en el bloque de diseño, placebo y evaluación única en el de prueba
5. Fase 4: Control positivo con carry (cuando estén las tasas)
6. Fase 5: Comparación entre estrategias
7. Conclusión

## Fase 0: Librerías y parámetros

Esta fase solo prepara el entorno. Los datos de cada control se cargan en su propia fase, para que el bloque de prueba de cada uno no se vea antes de tiempo.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings
from scipy import stats
warnings.filterwarnings('ignore')
rng = np.random.default_rng(0)

## Fase 1: Poder estadístico

**Idea.** Decir "el neto no es significativo" solo tiene valor si, de haber existido una ganancia, la habríamos visto. Eso depende sobre todo de **cuántos años de datos** hay (no de cuántas operaciones): una estrategia con Sharpe anual *S* medida durante *T* años da un estadístico t ≈ *S*·√*T*, así que para pasar t = 2.8 (95% de confianza y 80% de poder) hace falta *S* ≥ 2.8 / √*T*.

**Qué calculamos.**
- El Sharpe mínimo detectable para cada una de nuestras muestras: horaria (~1.9 años de ventanas de prueba), diaria en el bloque de diseño (~8.9 años) y diaria completa (~15.8 años).
- Una simulación del **poder** (probabilidad de detectar) para distintos Sharpe verdaderos, que confirma la fórmula y muestra el caso intermedio.

Sirve de contexto para todo lo anterior y para decidir cuánta historia necesitan los controles de las Fases 2 y 4.

In [2]:
def sharpe_minimo(anios, z=2.8):
    """Sharpe anual mínimo detectable con 95% de confianza y 80% de poder (aprox. de muestra grande)."""
    return z / np.sqrt(anios)

def poder(sharpe, anios, n_sim=4000, alpha=0.05, vol_anual=0.075):
    """Prob. de que un t-test unilateral de la media mensual rechace 0 si el Sharpe verdadero es `sharpe`."""
    n = int(round(anios * 12)); sd = vol_anual / np.sqrt(12); mu = sharpe * vol_anual / 12
    x = rng.normal(mu, sd, (n_sim, n))
    t = x.mean(1) / (x.std(1, ddof=1) / np.sqrt(n))
    return (t > stats.t.ppf(1 - alpha / 2, n - 1)).mean()

In [3]:
muestras = {'Horaria (16 ventanas de 30 días)': 1.9, 'Diaria, bloque de diseño 2011-2019': 8.9,
            'Diaria completa 2010-2026': 15.8, 'ETFs 1999-2019 (por ver)': 20.0}
display(pd.Series({k: round(sharpe_minimo(v), 2) for k, v in muestras.items()}, name='Sharpe mínimo detectable'))

Horaria (16 ventanas de 30 días)      2.03
Diaria, bloque de diseño 2011-2019    0.94
Diaria completa 2010-2026             0.70
ETFs 1999-2019 (por ver)              0.63
Name: Sharpe mínimo detectable, dtype: float64

In [4]:
sharpes = [0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0]
tabla = pd.DataFrame({f'{a} años': [poder(s, a) for s in sharpes] for a in (1.9, 5, 8.9, 15.8)}, index=sharpes)
tabla.index.name = 'Sharpe verdadero'
display(tabla.round(2))

,1.9 años,5 años,8.9 años,15.8 años
Sharpe verdadero,,,,
0.0,0.02,0.02,0.02,0.03
0.2,0.05,0.07,0.08,0.12
0.4,0.08,0.13,0.23,0.34
0.6,0.12,0.26,0.43,0.66
0.8,0.19,0.44,0.66,0.89
1.0,0.28,0.59,0.85,0.98
1.5,0.52,0.91,0.99,1.00
2.0,0.74,0.99,1.00,1.00


## Fase 2: Control positivo con ETFs (momentum): datos y regla

**Por qué este control.** El momentum de industrias (comprar los sectores que más han subido en el último año y vender los que más han bajado) es una de las primas más estudiadas en renta variable. Si nuestro procedimiento no encuentra una ganancia aquí, o el control no es tan claro como se dice, o nuestro método tiene un problema.

**Universo.** Los 9 ETFs de sectores de EE. UU. de SPDR (XLB materiales, XLE energía, XLF financiero, XLI industria, XLK tecnología, XLP consumo básico, XLU servicios públicos, XLV salud, XLY consumo discrecional), que existen desde diciembre de 1998. Los otros dos sectores (XLC, XLRE) se crearon después y no se incluyen para mantener un universo constante.

**Regla fijada de antemano.** La misma que en *Momentum-FX* para que la comparación sea directa: lookback de 12 meses, salto de 1, **3 largos y 3 cortos**, pesos iguales, rebalanceo mensual. No se ajustará nada.

**Sin bloques.** Como no hay parámetros que ajustar, evaluamos toda la muestra (más poder estadístico) y la desglosamos por subperiodos.

**Precios.** Precios de cierre ajustados por dividendos y splits. Se excluye el mes en curso (incompleto).

*Entrega:* `px` (precios), `mom_etf` y `W_etf` (retorno bruto mensual, rotación y pesos), con verificaciones.

In [5]:
import yfinance as yf
sectores = {'Materiales': 'XLB', 'Energia': 'XLE', 'Financiero': 'XLF', 'Industria': 'XLI', 'Tecnologia': 'XLK',
            'Consumo_basico': 'XLP', 'Servicios_publicos': 'XLU', 'Salud': 'XLV', 'Consumo_discrecional': 'XLY'}
raw_etf = yf.download(list(sectores.values()), start='1998-12-01', interval='1d', progress=False, auto_adjust=True)['Close']
n2t = {v: k for k, v in sectores.items()}
px = raw_etf.rename(columns=n2t)[list(sectores)]
if px.index.tz is not None:
    px.index = px.index.tz_localize(None)
px = px[px.index < pd.Timestamp('2026-11-01')].dropna()          # sin el mes en curso (incompleto)
print(px.shape, px.index.min().date(), '->', px.index.max().date())
print("Datos faltantes:", int(raw_etf.isna().sum().sum()), "(los anteriores al inicio de cada ETF)")

(7007, 9) 1998-12-22 -> 2026-10-30
Datos faltantes: 0 (los anteriores al inicio de cada ETF)


In [6]:
def momentum_mensual(px, lookback=12, skip=1, n_side=3):
    """Largo en las n_side series con mayor retorno [t-lookback, t-skip], corto en las n_side con menor retorno."""
    pm = px.groupby(px.index.to_period('M')).last()                  # precio de cierre de cada mes
    lp = np.log(pm); r = lp.diff()                                   # r[m]: retorno del mes m
    S = lp.shift(skip) - lp.shift(lookback)                          # señal conocida al cierre del mes m
    rank = S.rank(axis=1, method='first', ascending=False)
    W = pd.DataFrame(0.0, index=S.index, columns=S.columns)
    W[rank <= n_side] = 1.0 / n_side
    W[rank > S.notna().sum(axis=1).values[:, None] - n_side] = -1.0 / n_side
    W[S.isna().any(axis=1)] = 0.0                                    # sin historia suficiente: sin posición
    ret = (W.shift(1) * r).sum(axis=1)                               # los pesos del cierre m-1 ganan el retorno del mes m
    rot = (W - W.shift(1)).abs().sum(axis=1)                         # cuánto se opera cada mes
    out = pd.DataFrame({'ret_bruto': ret, 'rotacion': rot})
    return out.iloc[lookback + 1:], W

mom_etf, W_etf = momentum_mensual(px)

In [8]:
Wv = W_etf.loc[mom_etf.index]
print("Meses del portafolio:", len(mom_etf), "|", mom_etf.index.min(), "->", mom_etf.index.max())
print("Suma de pesos (debe ser 0):", Wv.sum(axis=1).abs().max())
print("Exposición bruta (debe ser 2):", Wv.abs().sum(axis=1).round(6).unique())
print("Largas / cortas por mes (3 / 3):", (Wv > 0).sum(axis=1).unique(), (Wv < 0).sum(axis=1).unique())
print("Rotación media mensual (sobre exposición 2):", mom_etf.rotacion.mean().round(3))
print("\nFracción de meses largo / corto, por sector:")
print(pd.DataFrame({'largo': (Wv > 0).mean(), 'corto': (Wv < 0).mean()}).round(2))

Meses del portafolio: 322 | 2000-01 -> 2026-10
Suma de pesos (debe ser 0): 5.551115123125783e-17
Exposición bruta (debe ser 2): [2.]
Largas / cortas por mes (3 / 3): [3] [3]
Rotación media mensual (sobre exposición 2): 0.87

Fracción de meses largo / corto, por sector:
                      largo  corto
Ticker                            
Materiales             0.33   0.35
Energia                0.34   0.42
Financiero             0.29   0.38
Industria              0.30   0.21
Tecnologia             0.44   0.26
Consumo_basico         0.25   0.39
Servicios_publicos     0.35   0.38
Salud                  0.31   0.36
Consumo_discrecional   0.39   0.26


## Fase 3: ETFs, resultados y placebo

Medimos cómo habría funcionado la regla (12-1, 3 + 3, mensual) en toda la muestra 2000–2026, **sin cambiarla**. Se reportan:
- **Retorno, volatilidad, Sharpe, estadístico t y caída máxima**, con costos de 0, 1, 3, 5 y 10 pb por pata. Los ETFs de sectores son de los más líquidos del mercado, así que 1–3 pb es un supuesto razonable y 5–10 pb es el escenario pesimista.
- **Resultados por subperiodo** (2000–2009, 2010–2019, 2020–2026), para ver si el resultado es estable o depende de una época.
- **Placebo de permutación:** se arman 2,000 portafolios que cada mes eligen 3 sectores largos y 3 cortos **al azar**, y se compara el Sharpe de la regla con la distribución del azar. Responde si el resultado es distinguible de elegir sectores sin criterio.

**Contexto del poder.** Con 26.8 años, el Sharpe mínimo detectable es ≈ 0.54 (Fase 1). Una prima de 0.3–0.5 podría existir y no ser significativa aquí; lo diremos con claridad si ocurre.

In [10]:
def metricas(r):
    mu, sd = r.mean() * 12, r.std() * np.sqrt(12)
    cum = r.cumsum()
    return pd.Series({'ret_anual': mu, 'vol_anual': sd, 'sharpe': mu / sd,
                      't': r.mean() / (r.std() / np.sqrt(len(r))),
                      'caida_max': (cum - cum.cummax()).min(), 'meses>0': (r > 0).mean()})

filas = {f'{b} pb por pata': metricas(mom_etf.ret_bruto - mom_etf.rotacion * b / 1e4) for b in (0, 1, 3, 5, 10)}
display(pd.DataFrame(filas).T.round(3))

,ret_anual,vol_anual,sharpe,t,caida_max,meses>0
0 pb por pata,0.016,0.14,0.115,0.596,-0.488,0.528
1 pb por pata,0.015,0.14,0.108,0.557,-0.498,0.528
3 pb por pata,0.013,0.14,0.093,0.480,-0.518,0.525
5 pb por pata,0.011,0.14,0.078,0.402,-0.538,0.522
10 pb por pata,0.006,0.14,0.040,0.209,-0.589,0.512


In [11]:
neto3 = mom_etf.ret_bruto - mom_etf.rotacion * 3 / 1e4
per = {'2000-2009': (2000, 2009), '2010-2019': (2010, 2019), '2020-2026': (2020, 2026)}
display(pd.DataFrame({k: metricas(neto3[(neto3.index.year >= a) & (neto3.index.year <= b)])
                      for k, (a, b) in per.items()}).T.round(3))

,ret_anual,vol_anual,sharpe,t,caida_max,meses>0
2000-2009,0.005,0.160,0.029,0.091,-0.324,0.500
2010-2019,-0.021,0.106,-0.199,-0.630,-0.340,0.525
2020-2026,0.075,0.151,0.497,1.299,-0.226,0.561


In [12]:
pm = px.groupby(px.index.to_period('M')).last()
R = np.log(pm).diff().loc[mom_etf.index].values                      # retornos mensuales de cada sector

def placebo_sharpes(R, n_sim=2000, n_side=3, seed=1):
    rg = np.random.default_rng(seed); T, K = R.shape
    ranks = rg.random((n_sim, T, K)).argsort(axis=2).argsort(axis=2)                 # orden aleatorio cada mes
    Wp = np.where(ranks < n_side, 1.0 / n_side, np.where(ranks >= K - n_side, -1.0 / n_side, 0.0))
    ret = (Wp[:, :-1, :] * R[None, 1:, :]).sum(axis=2)                                # pesos de t-1 ganan el retorno de t
    return ret.mean(1) * 12 / (ret.std(1, ddof=1) * np.sqrt(12))

sh_p = placebo_sharpes(R)
sh_real = metricas(mom_etf.ret_bruto).sharpe
print(f"Sharpe bruto de la regla: {sh_real:.3f}")
print(f"Placebo (azar): media {sh_p.mean():.3f}, desv. {sh_p.std():.3f}, percentil 95: {np.percentile(sh_p, 95):.3f}")
print(f"Fracción de placebos con Sharpe ≥ el de la regla (p-valor): {(sh_p >= sh_real).mean():.3f}")

Sharpe bruto de la regla: 0.115
Placebo (azar): media 0.002, desv. 0.198, percentil 95: 0.323
Fracción de placebos con Sharpe ≥ el de la regla (p-valor): 0.283
